# BootEA - Entity Alignment on DBP15K

> **Bootstrapping Entity Alignment with Knowledge Graph Embedding** - Zequn Sun, Wei Hu,
> Qingheng Zhang, Yuzhong Qu (*IJCAI 2018*)
> Source: https://www.ijcai.org/proceedings/2018/0611.pdf

A **self-contained** notebook (it imports nothing from an external package): the whole engine
is defined in the cells below, documented and structured. The same implementation is also
available in the `code/` package (`code/src/models/bootea.py`, class `BootEATrainer` in
`code/src/trainer.py`).

## The BootEA idea
BootEA learns *alignment-oriented* KG embeddings and self-trains:

1. **AlignE embedding**: a TransE energy `f(h,r,t) = ||h + r - t||` with a **limit-based**
   loss (absolute margins `gamma1`, `gamma2`) and entities constrained to the unit sphere.
2. **Epsilon-truncated negatives**: a triple is corrupted with the entity's **nearest same-KG
   neighbours** (hard negatives), refreshed periodically.
3. **Alignment by swapping**: for an aligned pair `(e1, e2)`, we swap `e1` and `e2` in their
   respective triples (`(e1,r,t) -> (e2,r,t)`), generating *aligned triples* that pull their
   embeddings together - the core mechanism of BootEA.
4. **Editable bootstrapping**: every few epochs a **mutual one-to-one matching (MWGM)**,
   recomputed from scratch, proposes new pairs (never accumulated => it corrects its own errors).

## Metrics: **MRR, Hit@1, Hit@5, Hit@10** (+ CSLS at evaluation).

## Reported results (DBP15K, 30% seed)
| Pair | Hit@1 | Hit@10 | MRR |
|------|------:|-------:|----:|
| zh_en | 62.94 | 84.75 | 0.703 |
| ja_en | 62.23 | 85.39 | 0.701 |
| fr_en | 65.30 | 87.44 | 0.731 |

---
## 1. Environment, imports and style

In [ ]:
import os, sys, csv, time, random, logging
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from collections import Counter

import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm          # plain terminal tqdm (not the notebook widget)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project root :", PROJECT_ROOT)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

SEED = 2024
set_seed(SEED)
torch.backends.cudnn.benchmark = True

### 1.1 Modern dark theme for the figures (identical to `code/src/utils/plotting.py`)

In [ ]:
"""Modern dark-theme plotting helpers.

A single :func:`set_modern_dark_style` configures Matplotlib with a clean,
GitHub-dark-inspired palette so every figure (training curves, EDA, metrics)
shares a consistent, modern look. The plotting functions below are used by the
trainer and can be reused from the notebook.
"""


# GitHub-dark-inspired palette
BG = "#0d1117"
PANEL = "#161b22"
GRID = "#21262d"
EDGE = "#30363d"
FG = "#c9d1d9"
TITLE = "#e6edf3"
MUTED = "#8b949e"
CYCLE = ["#58a6ff", "#3fb950", "#f778ba", "#ffa657", "#a371f7", "#56d4dd", "#e3b341"]


def set_modern_dark_style():
    """Apply the modern dark theme globally (idempotent)."""
    mpl.rcParams.update({
        "figure.facecolor": BG,
        "figure.edgecolor": BG,
        "savefig.facecolor": BG,
        "savefig.edgecolor": BG,
        "axes.facecolor": PANEL,
        "axes.edgecolor": EDGE,
        "axes.labelcolor": FG,
        "axes.titlecolor": TITLE,
        "axes.titleweight": "bold",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.linewidth": 1.0,
        "grid.color": GRID,
        "grid.linestyle": "--",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": FG,
        "legend.facecolor": PANEL,
        "legend.edgecolor": EDGE,
        "legend.framealpha": 0.9,
        "lines.linewidth": 2.2,
        "lines.markersize": 5,
        "lines.solid_capstyle": "round",
        "font.size": 11,
        "figure.dpi": 120,
        "axes.prop_cycle": mpl.cycler(color=CYCLE),
    })


def style_axes(ax, title=None, xlabel=None, ylabel=None):
    """Apply consistent spine/tick styling to an Axes."""
    if title:
        ax.set_title(title, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(length=0)
    return ax


def plot_loss_curves(loss_hist, ax=None, keys=("loss", "kge", "align", "pseudo")):
    """Plot per-epoch loss components from a list of dicts (with 'epoch')."""
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))
    ep = [h["epoch"] for h in loss_hist]
    labels = {"loss": "total", "kge": "kge (TransE)",
              "align": "align (seed)", "pseudo": "align (pseudo)"}
    for key in keys:
        ax.plot(ep, [h.get(key, 0.0) for h in loss_hist], label=labels.get(key, key))
    ax.legend()
    return style_axes(ax, "Training loss", "epoch", "loss")


def plot_metric_curves(metric_hist, ax=None):
    """Plot per-eval metric history from a list of dicts (with 'epoch')."""
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))
    ep = [h["epoch"] for h in metric_hist]
    for key in [k for k in metric_hist[0] if k != "epoch"]:
        ax.plot(ep, [h[key] for h in metric_hist], marker="o", label=key)
    ax.set_ylim(0, 1)
    ax.legend()
    return style_axes(ax, "Test metrics (avg direction)", "epoch", "score")

set_modern_dark_style()
print('Dark theme applique.')

---
## 2. Configuration (YAML) and logging
Driven by **`configs/bootea_dbp15k.yaml`**. Code identical to `code/src/utils/config.py` and `logger.py`.

In [ ]:
"""Configuration loading and run-directory bootstrap.

The whole experiment is driven by a single YAML file (see ``configs/``).
This module:
  * loads that YAML into a dotted-access namespace (``cfg.train.lr`` etc.),
  * creates a timestamped *run directory* under ``logging.output_dir``,
  * dumps the resolved config back to disk for reproducibility.

The logger is created separately by :mod:`src.utils.logger`.
"""




class Cfg(dict):
    """A ``dict`` that also supports attribute access, recursively.

    ``cfg.train.lr`` is equivalent to ``cfg["train"]["lr"]``. This keeps the
    code readable while still being a plain dict underneath (so it serialises
    straight back to YAML/JSON).
    """

    def __init__(self, d: dict | None = None):
        super().__init__()
        for k, v in (d or {}).items():
            self[k] = Cfg(v) if isinstance(v, dict) else v

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError as e:
            raise AttributeError(name) from e

    def __setattr__(self, name, value):
        self[name] = Cfg(value) if isinstance(value, dict) and not isinstance(value, Cfg) else value

    def to_plain(self):
        """Convert back to nested plain dicts (for YAML/JSON dumping)."""
        return {k: (v.to_plain() if isinstance(v, Cfg) else v) for k, v in self.items()}


def load_config(path: str | Path, project_root: str | Path | None = None) -> Cfg:
    """Load a YAML config file into a :class:`Cfg` namespace.

    ``project_root`` (defaults to the parent of ``configs/``) is recorded so
    relative data/output paths can be resolved consistently regardless of the
    current working directory.
    """
    path = Path(path).resolve()
    with open(path, "r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)
    cfg = Cfg(raw)
    root = Path(project_root).resolve() if project_root else path.parent.parent
    cfg._project_root = str(root)
    cfg._config_path = str(path)
    return cfg


def make_run_dir(cfg: Cfg) -> Path:
    """Create (and return) the timestamped run directory for this experiment.

    Layout::

        <output_dir>/<experiment.name>_<YYYYmmdd-HHMMSS>/
            training.txt          (full training log)
            config_used.yaml      (snapshot of the resolved config)
            model.pt / model_best.pt
            embeddings.pt
            metrics.csv / loss.csv
            *.png                 (loss & metric curves)
    """
    root = Path(cfg._project_root)
    out = root / cfg.logging.output_dir
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = out / f"{cfg.experiment.name}_{stamp}"
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg._run_dir = str(run_dir)

    # snapshot the exact config used for this run
    with open(run_dir / cfg.logging.config_dump, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg.to_plain(), f, sort_keys=False, allow_unicode=True)
    return run_dir

In [ ]:
"""Logging setup: write simultaneously to the console and to ``training.txt``."""




def get_logger(cfg: Cfg, run_dir: Path) -> logging.Logger:
    """Return a logger writing to both stdout and ``training.txt``.

    Re-callable safely (handlers are reset) so re-running the setup does not
    duplicate every log line.
    """
    logger = logging.getLogger(cfg.experiment.name)
    logger.setLevel(getattr(logging, cfg.logging.log_level.upper(), logging.INFO))
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")

    fh = logging.FileHandler(run_dir / cfg.logging.log_file, mode="w", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger

In [ ]:
cfg = load_config(PROJECT_ROOT / "configs" / "bootea_dbp15k.yaml", project_root=PROJECT_ROOT)
cfg.experiment.seed = SEED
cfg.experiment.name = f"bootea_{cfg.data.lang}_{cfg.data.fold}"

run_dir = make_run_dir(cfg)
logger = get_logger(cfg, run_dir)
device = torch.device(cfg.experiment.device if torch.cuda.is_available() else "cpu")
logger.info(f"Run dir : {run_dir}")
logger.info(f"Device  : {device}")
print(yaml.safe_dump(cfg.to_plain(), sort_keys=False, allow_unicode=True))

---
## 3. DBP15K data

JAPE/MTransE format (`<lang>/mtranse/<fold>/`): entity/relation ids **disjoint** between KG1
and KG2, contiguous `0..N-1`. The code below is identical to `code/src/data.py` and adds, on
top of the readers/samplers, the BootEA-specific helpers: `Swapper` (aligned-triple
generation) and `DynamicTripleSampler` (epsilon-truncated negatives).

In [ ]:
"""DBP15K data loading + neighbourhood construction for NAEA.

Data layout (the clean JAPE / MTransE split we use, ``<lang>/mtranse/<fold>/``):

    ent_ids_1 / ent_ids_2 : "<id>\\t<uri>"   entities of KG1 (e.g. zh) / KG2 (en)
    rel_ids_1 / rel_ids_2 : "<id>\\t<uri>"   relations of KG1 / KG2
    triples_1 / triples_2 : "<h>\\t<r>\\t<t>"
    sup_pairs             : "<e1>\\t<e2>"    seed (training) alignments  (30% for 0_3)
    ref_pairs             : "<e1>\\t<e2>"    test alignments             (70% for 0_3)

Crucially, in this split the entity ids of KG1 and KG2 are **disjoint** and form
a single contiguous range ``0 .. num_entities-1``; likewise relation ids. That
lets us use one shared embedding table and index it directly.
"""




# --------------------------------------------------------------------------- #
#  Raw file readers
# --------------------------------------------------------------------------- #
def _read_id_uri(path: Path) -> dict[int, str]:
    d = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                d[int(parts[0])] = parts[1]
    return d


def _read_triples(path: Path) -> np.ndarray:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) >= 3:
                rows.append((int(p[0]), int(p[1]), int(p[2])))
    return np.asarray(rows, dtype=np.int64)


def _read_pairs(path: Path) -> np.ndarray:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) >= 2:
                rows.append((int(p[0]), int(p[1])))
    return np.asarray(rows, dtype=np.int64)


# --------------------------------------------------------------------------- #
#  Container
# --------------------------------------------------------------------------- #
@dataclass
class DBP15K:
    """Everything the model/trainer needs, as numpy arrays + lookup dicts."""

    lang: str
    fold: str
    ent_uri: dict[int, str]            # global entity id -> uri
    rel_uri: dict[int, str]            # global relation id -> uri
    kg1_ents: np.ndarray               # entity ids belonging to KG1
    kg2_ents: np.ndarray               # entity ids belonging to KG2
    triples1: np.ndarray               # (M1, 3)  KG1 triples
    triples2: np.ndarray               # (M2, 3)  KG2 triples
    train_pairs: np.ndarray            # (S, 2)   seed alignments
    test_pairs: np.ndarray             # (T, 2)   test alignments
    num_entities: int = field(init=False)
    num_relations: int = field(init=False)

    def __post_init__(self):
        self.num_entities = int(max(self.ent_uri) + 1)
        self.num_relations = int(max(self.rel_uri) + 1)

    @property
    def triples(self) -> np.ndarray:
        return np.concatenate([self.triples1, self.triples2], axis=0)

    def summary(self) -> dict:
        return {
            "lang": self.lang,
            "fold": self.fold,
            "num_entities": self.num_entities,
            "num_relations": self.num_relations,
            "|KG1 ents|": len(self.kg1_ents),
            "|KG2 ents|": len(self.kg2_ents),
            "|KG1 triples|": len(self.triples1),
            "|KG2 triples|": len(self.triples2),
            "train_pairs": len(self.train_pairs),
            "test_pairs": len(self.test_pairs),
        }


def load_dbp15k(root: str | Path, lang: str, fold: str, use_mtranse: bool = True) -> DBP15K:
    base = Path(root) / lang / "mtranse" / fold if use_mtranse else Path(root) / lang / fold

    ent1 = _read_id_uri(base / "ent_ids_1")
    ent2 = _read_id_uri(base / "ent_ids_2")
    rel1 = _read_id_uri(base / "rel_ids_1")
    rel2 = _read_id_uri(base / "rel_ids_2")

    ent_uri = {**ent1, **ent2}
    rel_uri = {**rel1, **rel2}

    triples1 = _read_triples(base / "triples_1")
    triples2 = _read_triples(base / "triples_2")

    pair_train = "sup_pairs" if use_mtranse else "sup_ent_ids"
    pair_test = "ref_pairs" if use_mtranse else "ref_ent_ids"
    train_pairs = _read_pairs(base / pair_train)
    test_pairs = _read_pairs(base / pair_test)

    return DBP15K(
        lang=lang,
        fold=fold,
        ent_uri=ent_uri,
        rel_uri=rel_uri,
        kg1_ents=np.asarray(sorted(ent1), dtype=np.int64),
        kg2_ents=np.asarray(sorted(ent2), dtype=np.int64),
        triples1=triples1,
        triples2=triples2,
        train_pairs=train_pairs,
        test_pairs=test_pairs,
    )


# --------------------------------------------------------------------------- #
#  Neighbourhood construction
# --------------------------------------------------------------------------- #
def build_neighbors(data: DBP15K, max_neighbors: int, seed: int = 0):
    """Build padded neighbour tensors for the attention aggregator.

    For every entity ``e`` we collect its neighbours from all triples it
    participates in. A neighbour is a *(relation, entity, direction)* triplet:

      * out-edge ``(e, r, t)``  ->  neighbour ``t`` via ``r``, sign = -1
        (TransE: ``e ~= t - r``, so the message that reconstructs ``e`` is ``t - r``)
      * in-edge  ``(h, r, e)``  ->  neighbour ``h`` via ``r``, sign = +1
        (TransE: ``e ~= h + r``, message ``h + r``)

    Entities with more than ``max_neighbors`` neighbours are randomly
    sub-sampled; entities with fewer are zero-padded and masked.

    Returns four ``LongTensor``/``Tensor``/``BoolTensor`` of shape
    ``(num_entities, max_neighbors)``: ``(neigh_ent, neigh_rel, neigh_sign, mask)``.
    """
    rng = random.Random(seed)
    N = data.num_entities
    adj: list[list[tuple[int, int, int]]] = [[] for _ in range(N)]

    for arr in (data.triples1, data.triples2):
        for h, r, t in arr:
            adj[h].append((int(r), int(t), -1))   # out-edge:  e=h, message t - r
            adj[t].append((int(r), int(h), +1))   # in-edge:   e=t, message h + r

    K = max_neighbors
    neigh_ent = np.zeros((N, K), dtype=np.int64)
    neigh_rel = np.zeros((N, K), dtype=np.int64)
    neigh_sign = np.zeros((N, K), dtype=np.float32)
    mask = np.zeros((N, K), dtype=bool)

    for e in range(N):
        nbrs = adj[e]
        if len(nbrs) > K:
            nbrs = rng.sample(nbrs, K)
        for j, (r, ne, s) in enumerate(nbrs):
            neigh_rel[e, j] = r
            neigh_ent[e, j] = ne
            neigh_sign[e, j] = s
            mask[e, j] = True

    return (
        torch.from_numpy(neigh_ent),
        torch.from_numpy(neigh_rel),
        torch.from_numpy(neigh_sign),
        torch.from_numpy(mask),
    )


# --------------------------------------------------------------------------- #
#  Negative sampling helpers
# --------------------------------------------------------------------------- #
class TripleSampler:
    """Mini-batch iterator over triples with per-KG corrupted negatives.

    Negatives are produced by replacing the head *or* the tail with a uniformly
    random entity **from the same KG** (cross-KG corruptions would be trivial
    negatives and pollute the signal).
    """

    def __init__(self, data: DBP15K, device, neg: int = 5):
        self.device = device
        self.neg = neg
        t1 = torch.from_numpy(data.triples1)
        t2 = torch.from_numpy(data.triples2)
        kg = torch.cat([torch.zeros(len(t1), dtype=torch.long),
                        torch.ones(len(t2), dtype=torch.long)])
        self.triples = torch.cat([t1, t2], 0).to(device)        # (M,3)
        self.kg = kg.to(device)                                  # (M,)
        self.kg1_ents = torch.from_numpy(data.kg1_ents).to(device)
        self.kg2_ents = torch.from_numpy(data.kg2_ents).to(device)

    def __len__(self):
        return len(self.triples)

    def _rand_same_kg(self, kg_flags: torch.Tensor) -> torch.Tensor:
        """A random entity from the same KG as each row of ``kg_flags``."""
        n = kg_flags.shape[0]
        r1 = self.kg1_ents[torch.randint(len(self.kg1_ents), (n,), device=self.device)]
        r2 = self.kg2_ents[torch.randint(len(self.kg2_ents), (n,), device=self.device)]
        return torch.where(kg_flags == 0, r1, r2)

    def batches(self, batch_size: int, shuffle: bool = True):
        M = len(self.triples)
        order = torch.randperm(M, device=self.device) if shuffle else torch.arange(M, device=self.device)
        for s in range(0, M, batch_size):
            idx = order[s:s + batch_size]
            pos = self.triples[idx]                              # (B,3)
            kg = self.kg[idx]                                    # (B,)
            B = pos.shape[0]
            # repeat for `neg` corruptions
            pos_r = pos.repeat(self.neg, 1)                      # (B*neg,3)
            kg_r = kg.repeat(self.neg)
            neg = pos_r.clone()
            corrupt_head = torch.rand(B * self.neg, device=self.device) < 0.5
            rnd = self._rand_same_kg(kg_r)
            neg[:, 0] = torch.where(corrupt_head, rnd, neg[:, 0])
            neg[:, 2] = torch.where(~corrupt_head, rnd, neg[:, 2])
            yield pos, neg, kg


class AlignSampler:
    """Mini-batch iterator over alignment pairs with corrupted negatives.

    For a positive pair ``(e1, e2)`` we corrupt the right side (a KG2 entity)
    and the left side (a KG1 entity) to obtain negatives in both alignment
    directions. ``set_pairs`` lets the trainer inject bootstrapped pseudo-pairs.

    **Hard (nearest-neighbour) negatives.** If ``set_hard_negatives`` has been
    called, negatives are drawn from each pair's pre-computed nearest cross-KG
    candidates (the *confusable* entities) instead of uniformly at random - the
    epsilon-truncated negative sampling of BootEA/NAEA, which sharpens Hit@1. Any
    candidate that equals the gold target is replaced by a random one.
    """

    def __init__(self, data: DBP15K, device, neg: int = 5):
        self.device = device
        self.neg = neg
        self.kg1_ents = torch.from_numpy(data.kg1_ents).to(device)
        self.kg2_ents = torch.from_numpy(data.kg2_ents).to(device)
        self.hard_r = None          # (S, C) nearest KG2 candidates per pair (corrupt right)
        self.hard_l = None          # (S, C) nearest KG1 candidates per pair (corrupt left)
        self.set_pairs(data.train_pairs)

    def set_pairs(self, pairs):
        if isinstance(pairs, np.ndarray):
            pairs = torch.from_numpy(pairs)
        self.pairs = pairs.to(self.device).long()
        self.hard_r = self.hard_l = None    # invalidate stale hard-negative tables

    def set_hard_negatives(self, hard_r, hard_l):
        """``hard_r``/``hard_l``: (S, C) LongTensors aligned with ``self.pairs``."""
        self.hard_r = hard_r.to(self.device)
        self.hard_l = hard_l.to(self.device)

    def __len__(self):
        return len(self.pairs)

    def _sample_hard(self, table, rows, n, gold):
        """Pick ``n`` candidates per row from ``table[rows]``, avoiding ``gold``."""
        cand = table[rows]                                       # (B, C)
        C = cand.shape[1]
        sel = torch.randint(C, (cand.shape[0], n), device=self.device)
        out = torch.gather(cand, 1, sel).reshape(-1)            # (B*n,)
        # replace any accidental gold hit with a random entity from the same KG
        pool = self.kg2_ents if table is self.hard_r else self.kg1_ents
        clash = out == gold
        if clash.any():
            out[clash] = pool[torch.randint(len(pool), (int(clash.sum()),), device=self.device)]
        return out

    def batches(self, batch_size: int, shuffle: bool = True):
        S = len(self.pairs)
        order = torch.randperm(S, device=self.device) if shuffle else torch.arange(S, device=self.device)
        for s in range(0, S, batch_size):
            idx = order[s:s + batch_size]
            pos = self.pairs[idx]                                # (B,2)
            B = pos.shape[0]
            n = self.neg
            e1 = pos[:, 0].repeat_interleave(n)
            e2 = pos[:, 1].repeat_interleave(n)
            if self.hard_r is not None and self.hard_l is not None:
                neg_r = self._sample_hard(self.hard_r, idx, n, e2)
                neg_l = self._sample_hard(self.hard_l, idx, n, e1)
            else:
                neg_r = self.kg2_ents[torch.randint(len(self.kg2_ents), (B * n,), device=self.device)]
                neg_l = self.kg1_ents[torch.randint(len(self.kg1_ents), (B * n,), device=self.device)]
            yield pos, (e1, e2, neg_l, neg_r)


# =========================================================================== #
#  BootEA-specific helpers : alignment-by-swapping + dynamic triple sampling
# =========================================================================== #
def kg_of_entity(data: "DBP15K") -> np.ndarray:
    """Return an array ``kg[e] in {0, 1}`` giving the KG each entity belongs to."""
    kg = np.zeros(data.num_entities, dtype=np.int64)
    kg[data.kg2_ents] = 1
    return kg


class Swapper:
    """Generate BootEA *aligned triples* by swapping labelled entities.

    For a labelled pair ``(a, b)`` we substitute ``b`` for ``a`` in each of
    ``a``'s triples and vice-versa, e.g. ``(a, r, t) -> (b, r, t)``. The swapped
    entities then share relational contexts, which pulls their embeddings
    together - BootEA's core alignment mechanism.

    Per-entity triple lists are precomputed once; :meth:`generate` is then cheap
    enough to be re-run every bootstrapping round on the (growing) labelled set.
    """

    def __init__(self, data: "DBP15K"):
        triples = np.concatenate([data.triples1, data.triples2], axis=0)
        self.triples = triples
        n = data.num_entities
        self.as_head = [[] for _ in range(n)]
        self.as_tail = [[] for _ in range(n)]
        for i, (h, _r, t) in enumerate(triples):
            self.as_head[int(h)].append(i)
            self.as_tail[int(t)].append(i)

    def generate(self, labeled_pairs, cap_per_role: int = 100) -> np.ndarray:
        """Return an ``(K, 3)`` array of swapped (h, r, t) triples for the pairs."""
        T = self.triples
        out = []
        for a, b in labeled_pairs:
            a, b = int(a), int(b)
            for src, dst in ((a, b), (b, a)):          # dst takes src's place
                for ti in self.as_head[src][:cap_per_role]:
                    out.append((dst, T[ti, 1], T[ti, 2]))
                for ti in self.as_tail[src][:cap_per_role]:
                    out.append((T[ti, 0], T[ti, 1], dst))
        if not out:
            return np.empty((0, 3), dtype=np.int64)
        return np.asarray(out, dtype=np.int64)


class DynamicTripleSampler:
    """Mini-batch sampler over a *mutable* set of triples (BootEA / AlignE).

    Unlike :class:`TripleSampler` (which fixes the per-KG split), this sampler
    determines each corrupted entity's KG from a global ``kg_of_ent`` array, so
    it correctly handles **swapped** triples whose head and tail may live in
    different KGs. Negatives are uniform within the corrupted position's KG, or
    **epsilon-truncated** (drawn from that entity's nearest same-KG neighbours) when a
    candidate table has been provided via :meth:`set_candidates`.
    """

    def __init__(self, device, kg_of_ent: np.ndarray, kg1_ents, kg2_ents, neg: int = 5):
        self.device = device
        self.neg = neg
        self.kg_of_ent = torch.as_tensor(kg_of_ent, device=device).long()
        self.kg1_ents = torch.as_tensor(kg1_ents, device=device).long()
        self.kg2_ents = torch.as_tensor(kg2_ents, device=device).long()
        self.cand = None            # (N, C) nearest same-KG neighbours, optional
        self.triples = torch.empty((0, 3), dtype=torch.long, device=device)

    def set_triples(self, triples):
        if isinstance(triples, np.ndarray):
            triples = torch.from_numpy(triples)
        self.triples = triples.to(self.device).long()

    def set_candidates(self, cand):
        """``cand`` : (N, C) LongTensor of nearest same-KG neighbours per entity."""
        self.cand = cand.to(self.device) if cand is not None else None

    def __len__(self):
        return len(self.triples)

    def _rand_same_kg(self, ent_ids):
        kg = self.kg_of_ent[ent_ids]
        n = ent_ids.shape[0]
        r1 = self.kg1_ents[torch.randint(len(self.kg1_ents), (n,), device=self.device)]
        r2 = self.kg2_ents[torch.randint(len(self.kg2_ents), (n,), device=self.device)]
        return torch.where(kg == 0, r1, r2)

    def _trunc_same_kg(self, ent_ids):
        """epsilon-truncated: a random one of each entity's nearest same-KG neighbours."""
        cand = self.cand[ent_ids]                              # (n, C)
        sel = torch.randint(cand.shape[1], (cand.shape[0], 1), device=self.device)
        return torch.gather(cand, 1, sel).squeeze(1)

    def batches(self, batch_size: int, shuffle: bool = True):
        M = len(self.triples)
        order = torch.randperm(M, device=self.device) if shuffle else torch.arange(M, device=self.device)
        for s in range(0, M, batch_size):
            idx = order[s:s + batch_size]
            pos = self.triples[idx]                            # (B,3)
            B = pos.shape[0]
            n = self.neg
            pos_r = pos.repeat(n, 1)                           # (B*n,3)
            neg = pos_r.clone()
            corrupt_head = torch.rand(B * n, device=self.device) < 0.5
            # entity currently occupying the position we corrupt
            orig = torch.where(corrupt_head, pos_r[:, 0], pos_r[:, 2])
            repl = self._trunc_same_kg(orig) if self.cand is not None else self._rand_same_kg(orig)
            neg[:, 0] = torch.where(corrupt_head, repl, neg[:, 0])
            neg[:, 2] = torch.where(~corrupt_head, repl, neg[:, 2])
            yield pos, neg

In [ ]:
data = load_dbp15k(PROJECT_ROOT / cfg.data.root, cfg.data.lang,
                   cfg.data.fold, cfg.data.use_mtranse_format)
logger.info(f"Data: {data.summary()}")
pd.DataFrame.from_dict(data.summary(), orient="index", columns=["valeur"])

### 3.1 A few aligned pairs (ground truth)

In [ ]:
rows = []
for e1, e2 in data.test_pairs[:8]:
    rows.append({"KG1 id": e1, "KG1 entity": data.ent_uri[e1].split("/")[-1],
                 "KG2 id": e2, "KG2 entity": data.ent_uri[e2].split("/")[-1]})
pd.DataFrame(rows)

### 3.2 Exploratory data analysis (EDA): degrees and relations

In [ ]:
deg = Counter()
for arr in (data.triples1, data.triples2):
    for h, r, t in arr:
        deg[h] += 1; deg[t] += 1
deg1 = np.array([deg[e] for e in data.kg1_ents])
deg2 = np.array([deg[e] for e in data.kg2_ents])
rel_count1 = Counter(data.triples1[:, 1]); rel_count2 = Counter(data.triples2[:, 1])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].hist(np.clip(deg1, 0, 60), bins=40, alpha=0.85, label="KG1")
axes[0].hist(np.clip(deg2, 0, 60), bins=40, alpha=0.7, label="KG2")
style_axes(axes[0], "Degree distribution (clip 60)", "degree", "frequency"); axes[0].legend()
top1 = rel_count1.most_common(15)
axes[1].barh([data.rel_uri[r].split("/")[-1][:18] for r, _ in top1][::-1],
             [c for _, c in top1][::-1], color=CYCLE[0])
style_axes(axes[1], f"Top relations KG1 ({cfg.data.lang.split('_')[0]})", "frequency", None)
top2 = rel_count2.most_common(15)
axes[2].barh([data.rel_uri[r].split("/")[-1][:18] for r, _ in top2][::-1],
             [c for _, c in top2][::-1], color=CYCLE[2])
style_axes(axes[2], "Top relations KG2 (en)", "frequency", None)
plt.tight_layout(); plt.show()
print(f"mean degree KG1={deg1.mean():.1f} KG2={deg2.mean():.1f} | max KG1={deg1.max()} KG2={deg2.max()}")

### 3.3 A look at swapping (aligned triples)
We illustrate the BootEA mechanism: starting from the *seed* pairs, we generate aligned
triples by swapping the entities. These triples are added to the training set.

In [ ]:
swapper = Swapper(data)
swapped = swapper.generate(data.train_pairs[:50], cap_per_role=cfg.train.swapping.cap_per_role)
print(f"50 seed pairs -> {len(swapped)} aligned triples generated (example; full seed used at train time).")
ex = swapped[:5]
pd.DataFrame([{"h": data.ent_uri[h].split('/')[-1], "r": data.rel_uri[r].split('/')[-1],
               "t": data.ent_uri[t].split('/')[-1]} for h, r, t in ex])

---
## 4. The BootEA model (AlignE)

`encode(e)` = entity embedding (L2-normalised) - **no** neighbourhood aggregation (that is
what distinguishes BootEA from NAEA). `triple_score = ||h + r - t||`.

**Limit-based embedding loss** (absolute margins):
`O_e = sum_pos relu(f - gamma1) + mu * sum_neg relu(gamma2 - f)`.

**Contrastive alignment loss** (the Hit@1 driver): `alignment_loss` pulls aligned pairs below
`gamma1` and pushes **hard negatives** (nearest cross-KG neighbours) beyond `gamma2`. This
function (shared with NAEA, defined in `code/src/models/naea.py`) works on any model exposing
`encode()`. Code identical to `code/src/models/bootea.py`.

In [ ]:
"""The BootEA model.

BootEA (Sun, Hu, Zhang, Qu - "Bootstrapping Entity Alignment with Knowledge
Graph Embedding", IJCAI 2018) learns alignment-oriented KG embeddings:

1. **Alignment-oriented embedding (AlignE).** A TransE energy
   ``f(h, r, t) = ||h + r - t||`` trained with a **limit-based** objective
   (absolute margins) rather than a relative margin:

       O_e = sum_{tau in D+} [ f(tau) - gamma1 ]+  +  mu . sum_{tau' in D-} [ gamma2 - f(tau') ]+

   Entity embeddings are constrained to the unit sphere (L2-normalised);
   relation embeddings are free. Negatives use **epsilon-truncated** sampling
   (corrupt with one of the entity's nearest same-KG neighbours).

2. **Alignment by swapping.** For a labelled pair ``(e1, e2)`` BootEA generates
   *aligned triples* by swapping ``e1`` and ``e2`` in each other's triples
   (handled in the trainer / data module). Because the swapped entities then
   share relational contexts, their embeddings are pulled together - this is
   BootEA's core alignment mechanism, complemented here by a light limit-based
   pull on the labelled pairs.

3. **Bootstrapping.** An editable, recomputed maximum-weight (1-to-1) matching
   over the unlabelled pool proposes new alignments each round (in the trainer).

The representation used for **alignment** and **evaluation** is simply the
(L2-normalised) entity embedding - there is no neighbourhood aggregation here
(that is what distinguishes BootEA from NAEA).
"""



class BootEA(nn.Module):
    def __init__(
        self,
        num_entities: int,
        num_relations: int,
        embed_dim: int,
        init: str = "xavier",
        normalize_embeddings: bool = True,
    ):
        super().__init__()
        self.dim = embed_dim
        self.normalize_embeddings = normalize_embeddings
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        self.rel_emb = nn.Embedding(num_relations, embed_dim)
        self._init_params(init)

    def _init_params(self, init: str):
        if init == "xavier":
            nn.init.xavier_uniform_(self.ent_emb.weight)
            nn.init.xavier_uniform_(self.rel_emb.weight)
        else:
            nn.init.normal_(self.ent_emb.weight, std=0.02)
            nn.init.normal_(self.rel_emb.weight, std=0.02)

    # ------------------------------------------------------------------ #
    #  Embeddings
    # ------------------------------------------------------------------ #
    def _ent(self, idx):
        e = self.ent_emb(idx)
        return F.normalize(e, dim=-1) if self.normalize_embeddings else e

    def _rel(self, idx):
        # relations are left unconstrained in AlignE
        return self.rel_emb(idx)

    def triple_score(self, triples: torch.Tensor) -> torch.Tensor:
        """``||h + r - t||_2`` for a batch of (h, r, t). Lower = more plausible."""
        h = self._ent(triples[:, 0])
        r = self._rel(triples[:, 1])
        t = self._ent(triples[:, 2])
        return torch.norm(h + r - t, p=2, dim=-1)

    def encode(self, ent_idx: torch.Tensor) -> torch.Tensor:
        """Representation used for alignment / evaluation: the (normalised) entity embedding."""
        e = self.ent_emb(ent_idx)
        return F.normalize(e, dim=-1) if self.normalize_embeddings else e

    @torch.no_grad()
    def encode_all(self, ent_ids: torch.Tensor, chunk: int = 8192) -> torch.Tensor:
        self.eval()
        outs = [self.encode(ent_ids[s:s + chunk]) for s in range(0, len(ent_ids), chunk)]
        return torch.cat(outs, 0)


# --------------------------------------------------------------------------- #
#  Loss functions
# --------------------------------------------------------------------------- #
def limit_based_triple_loss(pos_score, neg_score, pos_margin: float,
                            neg_margin: float, neg_weight: float) -> torch.Tensor:
    """AlignE limit-based objective (absolute margins).

      * positives pulled **below** ``pos_margin`` : ``[ f(tau) - gamma1 ]+``
      * negatives pushed **above** ``neg_margin`` : ``[ gamma2 - f(tau') ]+``
    """
    l_pos = F.relu(pos_score - pos_margin).mean()
    l_neg = F.relu(neg_margin - neg_score).mean()
    return l_pos + neg_weight * l_neg


def alignment_pull_loss(model: BootEA, e1, e2, pos_margin: float) -> torch.Tensor:
    """Light limit-based pull on labelled aligned pairs (complements swapping).

    Pulls ``d(z_e1, z_e2)`` below ``pos_margin`` (with normalised embeddings the
    distance is in ``[0, 2]``).
    """
    z1 = model.encode(e1)
    z2 = model.encode(e2)
    d = torch.norm(z1 - z2, p=2, dim=-1)
    return F.relu(d - pos_margin).mean()


# --- contrastive alignment loss (shared, from code/src/models/naea.py) ---
def alignment_loss(model, e1, e2, neg_l, neg_r,
                   pos_margin: float, neg_margin: float, neg_weight: float = 1.0) -> torch.Tensor:
    """**Limit-based** (absolute-margin) alignment loss - BootEA-style.

    With L2-normalised ``z`` the distance ``d  in  [0, 2]``, so a *relative* margin
    can never saturate and collapses the space (especially with hard negatives).
    The limit-based objective instead sets absolute targets that **saturate**:

      * pull positives **below** ``pos_margin``  : ``[ d(z_e1,z_e2) - gamma1 ]+``
      * push negatives **above** ``neg_margin``  : ``[ gamma2 - d(z_e1,z_negR) ]+`` (+ left side)

    Once a negative is far enough (``d >= gamma2``) its gradient is zero -> no runaway
    repulsion, even for nearest-neighbour (hard) negatives.
    """
    z1 = model.encode(e1)
    z2 = model.encode(e2)
    zr = model.encode(neg_r)
    zl = model.encode(neg_l)
    d_pos = torch.norm(z1 - z2, p=2, dim=-1)                     # (B*neg,)
    d_neg_r = torch.norm(z1 - zr, p=2, dim=-1)
    d_neg_l = torch.norm(zl - z2, p=2, dim=-1)
    l_pos = F.relu(d_pos - pos_margin).mean()
    l_neg = 0.5 * (F.relu(neg_margin - d_neg_r).mean() + F.relu(neg_margin - d_neg_l).mean())
    return l_pos + neg_weight * l_neg

In [ ]:
set_seed(cfg.experiment.seed)
model = BootEA(num_entities=data.num_entities, num_relations=data.num_relations,
               embed_dim=cfg.model.embed_dim, init=cfg.model.init,
               normalize_embeddings=cfg.model.normalize_embeddings).to(device)
logger.info(f"BootEA : {sum(p.numel() for p in model.parameters())/1e6:.2f}M parametres | dim={cfg.model.embed_dim}")

---
## 5. Metrics: MRR, Hit@k, CSLS (identical to `code/src/utils/metrics.py`)

In [ ]:
"""Entity-alignment evaluation: MRR and Hit@k, with cosine or CSLS scoring.

Protocol (standard for DBP15K): the test set is a list of gold pairs
``(e1_i, e2_i)``. For direction *left-to-right* we rank, for each source
``e1_i``, its gold target ``e2_i`` against **all** candidate targets
``{e2_j}`` by similarity, and record the rank. Hit@k = fraction of sources
whose gold rank <= k; MRR = mean of ``1 / rank``.

CSLS (Cross-domain Similarity Local Scaling, Lample et al. 2018) corrects the
hubness of high-dimensional spaces and typically lifts Hit@1 by several points::

    csls(x, y) = 2.cos(x, y) - r_T(x) - r_S(y)

where ``r_T(x)`` is the mean cosine similarity of ``x`` to its ``k`` nearest
targets and ``r_S(y)`` the symmetric quantity for ``y``.
"""



def _mean_topk_sim(sim: torch.Tensor, k: int, dim: int) -> torch.Tensor:
    """Mean of the top-``k`` similarities along ``dim`` (CSLS local scaling)."""
    k = min(k, sim.shape[dim])
    vals, _ = sim.topk(k, dim=dim)
    return vals.mean(dim=dim)


@torch.no_grad()
def _rank_metrics(sim: torch.Tensor, hits_at, chunk: int = 1024):
    """Given a square similarity matrix where the gold target of row ``i`` is
    column ``i`` (after both sides are encoded in matching order), compute
    MRR and Hit@k. Done in row-chunks to bound memory."""
    n = sim.shape[0]
    device = sim.device
    ranks = torch.empty(n, device=device)
    gold = torch.arange(n, device=device)
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        block = sim[s:e]                                         # (c, n)
        gold_sim = block[torch.arange(e - s, device=device), gold[s:e]].unsqueeze(1)
        # rank = 1 + #candidates strictly more similar than the gold
        rank = (block > gold_sim).sum(1) + 1
        ranks[s:e] = rank.float()
    out = {"MRR": (1.0 / ranks).mean().item()}
    for k in hits_at:
        out[f"Hit@{k}"] = (ranks <= k).float().mean().item()
    out["MeanRank"] = ranks.mean().item()
    return out


@torch.no_grad()
def evaluate_alignment(
    z_left: torch.Tensor,
    z_right: torch.Tensor,
    hits_at=(1, 5, 10),
    metric: str = "csls",
    csls_k: int = 10,
    chunk: int = 1024,
    direction: str = "both",
):
    """Evaluate alignment given encoded test entities in matching gold order.

    ``z_left[i]`` aligns to ``z_right[i]``. Embeddings are L2-normalised here so
    cosine == dot product. Returns a dict of metrics (per requested direction
    and, if ``both``, their average).
    """
    zl = torch.nn.functional.normalize(z_left, dim=-1)
    zr = torch.nn.functional.normalize(z_right, dim=-1)
    sim = zl @ zr.t()                                            # (n, n) cosine

    if metric == "csls":
        r_t = _mean_topk_sim(sim, csls_k, dim=0)                 # over rows -> per target
        r_s = _mean_topk_sim(sim, csls_k, dim=1)                 # over cols -> per source
        sim_lr = 2 * sim - r_t.unsqueeze(0) - r_s.unsqueeze(1)
        sim_rl = sim_lr.t()
    elif metric in ("cosine", "l2"):
        sim_lr = sim
        sim_rl = sim.t()
    else:
        raise ValueError(f"unknown metric {metric!r}")

    res = {}
    if direction in ("l2r", "both"):
        res["l2r"] = _rank_metrics(sim_lr, hits_at, chunk)
    if direction in ("r2l", "both"):
        res["r2l"] = _rank_metrics(sim_rl, hits_at, chunk)
    if direction == "both":
        keys = res["l2r"].keys()
        res["avg"] = {k: 0.5 * (res["l2r"][k] + res["r2l"][k]) for k in keys}
    return res


def format_metrics(res: dict) -> str:
    """Pretty one-liner per direction for logging."""
    lines = []
    for d, m in res.items():
        parts = [f"MRR={m['MRR']:.4f}"]
        parts += [f"{k}={v:.4f}" for k, v in m.items() if k.startswith("Hit@")]
        parts.append(f"MR={m['MeanRank']:.1f}")
        lines.append(f"[{d:>3}] " + " ".join(parts))
    return " | ".join(lines)

In [ ]:
test_left  = torch.from_numpy(data.test_pairs[:, 0]).to(device)
test_right = torch.from_numpy(data.test_pairs[:, 1]).to(device)
with torch.no_grad():
    z_l = model.encode_all(test_left); z_r = model.encode_all(test_right)
res0 = evaluate_alignment(z_l, z_r, hits_at=tuple(cfg.eval.hits_at), metric=cfg.eval.metric,
                          csls_k=cfg.eval.csls_k, chunk=cfg.eval.eval_chunk, direction=cfg.eval.direction)
print("Baseline (UNtrained model):")
print(format_metrics(res0))

---
## 6. Training (BootEA)

The `BootEATrainer` (identical to the class of the same name in `code/src/trainer.py`):
limit-based loss over the *mutable* triple set (base + swapping-aligned triples) with
epsilon-truncated negatives, an alignment pull, **editable MWGM bootstrapping**, evaluation +
early-stop, and full logging (`training.txt`, CSV, PNG curves, checkpoints, embeddings).
`tqdm` progress bars in the terminal.

In [ ]:
class BootEATrainer:
    """BootEA training loop (Sun et al., IJCAI 2018).

    Components:
      * **AlignE embedding** : limit-based TransE loss over a (mutable) triple set
        with **epsilon-truncated** negative sampling (nearest same-KG neighbours).
      * **Alignment by swapping** : seed-aligned pairs generate aligned triples
        (added once to the triple set) that calibrate the two embedding spaces.
      * **Contrastive alignment loss** : a limit-based, hard-negative contrastive
        objective on the labelled pairs (seed + bootstrapped). This is the
        reliable driver of Hit@1 and corresponds to BootEA's alignment likelihood.
      * **Editable MWGM bootstrapping** : recomputed mutual 1-to-1 matching each
        round feeds the *pseudo* alignment term (down-weighted, never accumulated).

    Note: swapping is restricted to the **gold seed** pairs (built once) so that
    erroneous bootstrapped pairs never corrupt the triple set - the cause of the
    collapse seen with naive swap-on-pseudo-labels.
    """

    def __init__(self, cfg, data: DBP15K, model: BootEA, run_dir: Path, logger):
        self.cfg = cfg
        self.data = data
        self.model = model
        self.run_dir = Path(run_dir)
        self.log = logger
        self.device = next(model.parameters()).device

        self.kg_of_ent = kg_of_entity(data)
        self.kg1_ents = torch.from_numpy(data.kg1_ents).to(self.device)
        self.kg2_ents = torch.from_numpy(data.kg2_ents).to(self.device)

        # triple sampler over base + seed-swapped triples (epsilon-truncated negatives)
        self.triple_sampler = DynamicTripleSampler(
            self.device, self.kg_of_ent, data.kg1_ents, data.kg2_ents, neg=cfg.train.neg_samples)
        self.swapper = Swapper(data)
        self.base_triples = np.concatenate([data.triples1, data.triples2], axis=0)
        self.labeled = data.train_pairs.copy()      # grows via bootstrapping (if swap_pseudo)
        self._rebuild_triples(self.labeled)

        # alignment: seed sampler (with hard negatives) + recomputed pseudo sampler
        self.align_sampler = AlignSampler(data, self.device, neg=cfg.train.neg_samples)
        self.pseudo_sampler = None
        self.n_pseudo = 0

        params = model.parameters()
        if cfg.train.optimizer.lower() == "adam":
            self.optimizer = torch.optim.Adam(params, lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        else:
            self.optimizer = torch.optim.SGD(params, lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        self.scheduler = None
        if str(cfg.train.get("lr_schedule", "none")).lower() == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, T_max=cfg.train.epochs, eta_min=cfg.train.lr * 0.02)

        self.test_left = torch.from_numpy(data.test_pairs[:, 0]).to(self.device)
        self.test_right = torch.from_numpy(data.test_pairs[:, 1]).to(self.device)

        self.loss_hist, self.metric_hist = [], []
        self.best_mrr, self.best_epoch, self.no_improve = -1.0, -1, 0

    # ------------------------------------------------------------------ #
    #  Triple set : base + aligned (swapped) triples from labelled pairs
    # ------------------------------------------------------------------ #
    def _rebuild_triples(self, pairs):
        """Set the triple set to base + swapped triples generated from ``pairs``.

        With ``swapping.swap_pseudo`` the labelled set grows via bootstrapping, so
        confident bootstrapped pairs also calibrate the embeddings (BootEA's core
        idea). Recomputed from scratch each round (editable) so wrong pairs are
        dropped as the matching is revised.
        """
        triples = self.base_triples
        if self.cfg.train.swapping.enabled and len(pairs):
            swapped = self.swapper.generate(pairs, cap_per_role=self.cfg.train.swapping.cap_per_role)
            if len(swapped):
                triples = np.concatenate([triples, swapped], axis=0)
        self.triple_sampler.set_triples(triples)
        return len(triples)

    # ------------------------------------------------------------------ #
    #  epsilon-truncated triple negatives (nearest same-KG neighbours)
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def refresh_truncated_candidates(self):
        C = self.cfg.train.eps_truncated.num_candidates
        self.model.eval()
        N = self.data.num_entities
        cand = torch.zeros((N, C), dtype=torch.long, device=self.device)
        for ids in (self.kg1_ents, self.kg2_ents):
            z = torch.nn.functional.normalize(self.model.encode_all(ids), dim=-1)
            for s in range(0, len(ids), 1024):
                sim = z[s:s + 1024] @ z.t()
                top = sim.topk(C + 1, dim=1).indices[:, 1:]      # drop self
                cand[ids[s:s + 1024]] = ids[top]
        self.triple_sampler.set_candidates(cand)

    # ------------------------------------------------------------------ #
    #  Hard (nearest cross-KG) negatives for the alignment loss
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def refresh_hard_negatives(self):
        C = self.cfg.train.hard_negatives.num_candidates
        self.model.eval()
        pairs = self.align_sampler.pairs
        left, right = pairs[:, 0], pairs[:, 1]
        z_left = torch.nn.functional.normalize(self.model.encode_all(left), dim=-1)
        z_right = torch.nn.functional.normalize(self.model.encode_all(right), dim=-1)
        z_kg1 = torch.nn.functional.normalize(self.model.encode_all(self.kg1_ents), dim=-1)
        z_kg2 = torch.nn.functional.normalize(self.model.encode_all(self.kg2_ents), dim=-1)

        def nearest(zq, zpool, pool_ids):
            out = torch.empty((zq.shape[0], C), dtype=torch.long, device=self.device)
            for s in range(0, zq.shape[0], 1024):
                top = (zq[s:s + 1024] @ zpool.t()).topk(C, dim=1).indices
                out[s:s + 1024] = pool_ids[top]
            return out

        hard_r = nearest(z_left, z_kg2, self.kg2_ents)
        hard_l = nearest(z_right, z_kg1, self.kg1_ents)
        self.align_sampler.set_hard_negatives(hard_r, hard_l)

    # ------------------------------------------------------------------ #
    #  One training epoch
    # ------------------------------------------------------------------ #
    def train_epoch(self, epoch: int):
        self.model.train()
        c = self.cfg.train
        align_batches = list(self.align_sampler.batches(c.align_batch_size))
        n_align = max(1, len(align_batches))
        pseudo_batches = (list(self.pseudo_sampler.batches(c.align_batch_size))
                          if self.pseudo_sampler is not None else [])
        n_pseudo = len(pseudo_batches)
        pw = c.bootstrap.pseudo_weight

        tot = {"loss": 0.0, "kge": 0.0, "align": 0.0, "pseudo": 0.0}
        steps = 0
        pbar = tqdm(self.triple_sampler.batches(c.batch_size),
                    total=(len(self.triple_sampler) + c.batch_size - 1) // c.batch_size,
                    desc=f"epoch {epoch:>4}", leave=False, ncols=100)
        for i, (pos, neg) in enumerate(pbar):
            self.optimizer.zero_grad()
            kge = limit_based_triple_loss(
                self.model.triple_score(pos), self.model.triple_score(neg),
                c.pos_margin_kge, c.neg_margin_kge, c.neg_weight_kge)

            _p, (e1, e2, nl, nr) = align_batches[i % n_align]
            align = alignment_loss(self.model, e1, e2, nl, nr,
                                   c.align_pos_margin, c.align_neg_margin, c.align_neg_weight)
            loss = kge + c.align_loss_weight * align

            pseudo_val = 0.0
            if n_pseudo:
                _q, (pe1, pe2, pnl, pnr) = pseudo_batches[i % n_pseudo]
                pseudo = alignment_loss(self.model, pe1, pe2, pnl, pnr,
                                        c.align_pos_margin, c.align_neg_margin, c.align_neg_weight)
                loss = loss + pw * pseudo
                pseudo_val = pseudo.item()

            loss.backward()
            if c.grad_clip and c.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), c.grad_clip)
            self.optimizer.step()

            tot["kge"] += kge.item(); tot["align"] += align.item()
            tot["pseudo"] += pseudo_val; tot["loss"] += loss.item()
            steps += 1
            pbar.set_postfix(loss=f"{loss.item():.3f}", kge=f"{kge.item():.3f}",
                             align=f"{align.item():.3f}", pseudo=f"{pseudo_val:.3f}")
        return {k: v / max(1, steps) for k, v in tot.items()}

    # ------------------------------------------------------------------ #
    #  Editable MWGM bootstrapping -> pseudo alignment pairs (recomputed)
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def bootstrap(self):
        bs = self.cfg.train.bootstrap
        self.model.eval()
        left, right = self.test_left, self.test_right
        zl = torch.nn.functional.normalize(self.model.encode_all(left), dim=-1)
        zr = torch.nn.functional.normalize(self.model.encode_all(right), dim=-1)
        cos = zl @ zr.t()
        k = self.cfg.eval.csls_k
        r_t = cos.topk(min(k, cos.shape[0]), dim=0).values.mean(0)
        r_s = cos.topk(min(k, cos.shape[1]), dim=1).values.mean(1)
        csls = 2 * cos - r_t.unsqueeze(0) - r_s.unsqueeze(1)

        idx = torch.arange(len(left), device=self.device)
        best_r = csls.argmax(1)
        best_l = csls.argmax(0)
        mutual = best_l[best_r] == idx
        conf = cos[idx, best_r]
        keep = mutual & (conf >= bs.threshold)
        cand = idx[keep]
        if len(cand) == 0:
            self.pseudo_sampler = None; self.n_pseudo = 0
            return 0
        order = torch.argsort(conf[cand], descending=True)
        cand = cand[order][: bs.max_add]
        used_r, pairs = set(), []
        for li, ri in zip(cand.tolist(), best_r[cand].tolist()):
            if ri in used_r:
                continue
            used_r.add(ri)
            pairs.append((int(left[li]), int(right[ri])))
        pairs = np.array(pairs, dtype=np.int64)
        if self.pseudo_sampler is None:
            self.pseudo_sampler = AlignSampler(self.data, self.device, neg=self.cfg.train.neg_samples)
        self.pseudo_sampler.set_pairs(pairs)              # REPLACE (editable, no accumulation)
        self.n_pseudo = len(pairs)

        # optionally feed confident pairs to the embedding via swapping (BootEA core)
        if self.cfg.train.swapping.get("swap_pseudo", False):
            seed = {int(a): int(b) for a, b in self.data.train_pairs}
            for a, b in pairs:
                seed.setdefault(int(a), int(b))
            self.labeled = np.array(list(seed.items()), dtype=np.int64)
            self._rebuild_triples(self.labeled)
        return len(pairs)

    # ------------------------------------------------------------------ #
    #  Evaluation / persistence / plots
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        ec = self.cfg.eval
        zl = self.model.encode_all(self.test_left)
        zr = self.model.encode_all(self.test_right)
        return evaluate_alignment(zl, zr, hits_at=tuple(ec.hits_at), metric=ec.metric,
                                    csls_k=ec.csls_k, chunk=ec.eval_chunk, direction=ec.direction)

    def save_checkpoint(self, name, epoch, res=None):
        torch.save({"epoch": epoch, "model_state": self.model.state_dict(),
                    "config": self.cfg.to_plain(), "metrics": res}, self.run_dir / name)

    def save_embeddings(self):
        torch.save({"ent_emb": self.model.ent_emb.weight.detach().cpu(),
                    "rel_emb": self.model.rel_emb.weight.detach().cpu()},
                   self.run_dir / self.cfg.logging.embeddings_name)

    def _append_csv(self, name, row, header_order=None):
        path = self.run_dir / name
        new = not path.exists()
        with open(path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=header_order or list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)

    def plot_curves(self):
        set_modern_dark_style()
        if self.loss_hist:
            fig, ax = plt.subplots(figsize=(8, 5))
            plot_loss_curves(self.loss_hist, ax=ax, keys=("loss", "kge", "align", "pseudo"))
            fig.tight_layout(); fig.savefig(self.run_dir / self.cfg.logging.plots.loss_curve); plt.close(fig)
        if self.metric_hist:
            fig, ax = plt.subplots(figsize=(8, 5))
            plot_metric_curves(self.metric_hist, ax=ax)
            fig.tight_layout(); fig.savefig(self.run_dir / self.cfg.logging.plots.metrics_curve); plt.close(fig)

    # ------------------------------------------------------------------ #
    #  Main loop
    # ------------------------------------------------------------------ #
    def fit(self):
        cfg = self.cfg
        self.log.info(f"Run directory: {self.run_dir}")
        self.log.info(f"Device: {self.device} | entities={self.data.num_entities} "
                      f"relations={self.data.num_relations} base_triples={len(self.base_triples)} "
                      f"triples(+seed-swap)={len(self.triple_sampler)} "
                      f"seed_pairs={len(self.data.train_pairs)} test_pairs={len(self.test_left)}")
        boot, eps, hn = cfg.train.bootstrap, cfg.train.eps_truncated, cfg.train.hard_negatives
        t0 = time.time()

        for epoch in range(1, cfg.train.epochs + 1):
            losses = self.train_epoch(epoch)
            losses["epoch"] = epoch
            self.loss_hist.append(losses)
            self._append_csv(cfg.logging.loss_csv, losses, ["epoch", "loss", "kge", "align", "pseudo"])
            msg = (f"epoch {epoch:>4}/{cfg.train.epochs} | loss={losses['loss']:.4f} "
                   f"(kge={losses['kge']:.4f} align={losses['align']:.4f} pseudo={losses['pseudo']:.4f})")

            if hn.enabled and epoch >= hn.start_epoch and (epoch - hn.start_epoch) % hn.refresh_every == 0:
                self.refresh_hard_negatives()
                msg += " | hard-neg refreshed"
            if eps.enabled and epoch >= eps.start_epoch and (epoch - eps.start_epoch) % eps.refresh_every == 0:
                self.refresh_truncated_candidates()
                msg += " | eps-trunc refreshed"
            if boot.enabled and epoch >= boot.start_epoch and (epoch - boot.start_epoch) % boot.every == 0:
                added = self.bootstrap()
                msg += f" | bootstrap: {added} pseudo-pairs (beta={boot.pseudo_weight})"

            if self.scheduler is not None:
                self.scheduler.step()
            self.log.info(msg)

            if epoch % cfg.eval.every == 0 or epoch == cfg.train.epochs:
                res = self.evaluate()
                self.log.info("           " + format_metrics(res))
                ref = res.get("avg", res.get("l2r"))
                self._append_csv(cfg.logging.metrics_csv, {"epoch": epoch, **{k: ref[k] for k in ref}})
                self.metric_hist.append({"epoch": epoch, "MRR": ref["MRR"],
                                         **{k: v for k, v in ref.items() if k.startswith("Hit@")}})
                self.plot_curves()
                if ref["MRR"] > self.best_mrr:
                    self.best_mrr, self.best_epoch, self.no_improve = ref["MRR"], epoch, 0
                    if cfg.logging.save_best:
                        self.save_checkpoint("model_best.pt", epoch, res)
                        self.log.info(f"           -> new best MRR={self.best_mrr:.4f} (saved model_best.pt)")
                else:
                    self.no_improve += 1
                patience = cfg.train.get("early_stop_patience", 0)
                if patience and self.no_improve >= patience:
                    self.log.info(f"           early stop: no MRR improvement for {patience} evals "
                                  f"(best={self.best_mrr:.4f} @ {self.best_epoch}).")
                    break

        if cfg.logging.save_last:
            self.save_checkpoint(cfg.logging.checkpoint_name, cfg.train.epochs)
        self.save_embeddings()
        self.plot_curves()
        self.log.info(f"Done in {(time.time()-t0)/60:.1f} min. Best MRR={self.best_mrr:.4f} @ epoch {self.best_epoch}.")
        return {"best_mrr": self.best_mrr, "best_epoch": self.best_epoch,
                "metric_hist": self.metric_hist, "loss_hist": self.loss_hist}

In [ ]:
set_seed(cfg.experiment.seed)
trainer = BootEATrainer(cfg, data, model, run_dir, logger)
history = trainer.fit()

---
## 7. Curves and results

In [ ]:
loss_df = pd.read_csv(run_dir / cfg.logging.loss_csv)
met_df  = pd.read_csv(run_dir / cfg.logging.metrics_csv)
display(met_df.tail(10))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_loss_curves(history["loss_hist"], ax=axes[0], keys=("loss", "kge", "align"))
plot_metric_curves(history["metric_hist"], ax=axes[1])
plt.tight_layout(); plt.show()
print(f"Best MRR = {history['best_mrr']:.4f} at epoch {history['best_epoch']}")

---
## 8. Final evaluation and qualitative analysis

In [ ]:
ckpt = torch.load(run_dir / "model_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
logger.info(f"Best checkpoint reloaded (epoch {ckpt['epoch']}).")
with torch.no_grad():
    z_l = model.encode_all(test_left); z_r = model.encode_all(test_right)
res = evaluate_alignment(z_l, z_r, hits_at=tuple(cfg.eval.hits_at), metric=cfg.eval.metric,
                         csls_k=cfg.eval.csls_k, chunk=cfg.eval.eval_chunk, direction="both")
print(format_metrics(res))

In [ ]:
with torch.no_grad():
    zl = F.normalize(model.encode_all(test_left), dim=-1)
    zr = F.normalize(model.encode_all(test_right), dim=-1)
    top = (zl @ zr.t()).topk(3, dim=1).indices.cpu().numpy()
rows = []
for i in range(10):
    status = "hit@1" if top[i][0] == i else ("top-3" if i in top[i] else "miss")
    rows.append({"source (KG1)": data.ent_uri[int(test_left[i])].split("/")[-1],
                 "or (KG2)": data.ent_uri[int(test_right[i])].split("/")[-1],
                 "top-1": data.ent_uri[int(test_right[top[i][0]])].split("/")[-1],
                 "top-2": data.ent_uri[int(test_right[top[i][1]])].split("/")[-1],
                 "top-3": data.ent_uri[int(test_right[top[i][2]])].split("/")[-1],
                 "statut": status})
pd.DataFrame(rows)

---
## 9. Saved artefacts

In [ ]:
print("Run dir :", run_dir, "\n")
for p in sorted(run_dir.iterdir()):
    print(f"  {p.name:<22} {p.stat().st_size/1024:8.1f} KB")

---
## 10. Comparison with the paper

In [ ]:
res_avg = res.get("avg", res.get("l2r"))
summary_df = pd.DataFrame([
    {"model": "BootEA (paper)", "Hit@1": 0.6294, "Hit@10": 0.8475, "MRR": 0.703},
    {"model": "This notebook",     "Hit@1": res_avg["Hit@1"], "Hit@10": res_avg["Hit@10"], "MRR": res_avg["MRR"]},
]).set_index("model")
display(summary_df.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
names = ["Hit@1", "Hit@10", "MRR"]; x = np.arange(len(names)); w = 0.38
ax.bar(x - w/2, [0.6294, 0.8475, 0.703], w, label="BootEA (paper)", color=CYCLE[5])
ax.bar(x + w/2, [res_avg["Hit@1"], res_avg["Hit@10"], res_avg["MRR"]], w, label="This notebook", color=CYCLE[0])
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 1)
style_axes(ax, f"BootEA paper vs notebook ({cfg.data.lang})", None, "score"); ax.legend()
plt.tight_layout(); plt.show()

---
## 11. Notes and references

**Implemented BootEA components**
- AlignE: TransE + limit-based loss (absolute margins gamma1, gamma2, mu); entities on the unit sphere.
- Epsilon-truncated negatives (nearest same-KG neighbours), refreshed periodically.
- Alignment by swapping (aligned triples) + a light pull on the labelled pairs.
- Editable bootstrapping via mutual one-to-one matching (MWGM), recomputed every round.
- CSLS at evaluation.

**Tuning levers**: `model.embed_dim` (paper 75), `train.pos/neg_margin_kge`,
`train.swapping.cap_per_role`, `train.bootstrap.threshold`, `train.eps_truncated.num_candidates`,
`train.lr_schedule: cosine`.

**References**
- Sun et al., BootEA, IJCAI 2018 - https://www.ijcai.org/proceedings/2018/0611.pdf
- Bordes et al., TransE, NeurIPS 2013.
- Lample et al., Word Translation Without Parallel Data (CSLS), ICLR 2018.
- Sun et al., A Benchmarking Study of EA (OpenEA), VLDB 2020.